In [ ]:
import subprocess
import sys
from pathlib import Path

import torch

CODE = next(
    path.parent.parent
    for path in Path("/kaggle/input").rglob("harness.py")
    if path.parent.name == "octlm"
)
OUT = Path("/kaggle/working/day9")
MODEL = OUT / "qwen"
SFT = OUT / "sft-1.pt"
GRPO = OUT / "grpo.pt"
OUT.mkdir(parents=True, exist_ok=True)
if not torch.cuda.is_available() or torch.cuda.get_device_capability(0) != (7, 5):
    raise RuntimeError("Select a Kaggle T4 GPU")
print("Code:", CODE, "GPU:", torch.cuda.get_device_name(0), flush=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "tokenizers==0.22.2", "jinja2>=3.1.6,<4"],
    check=True,
)


def run(module, stage, out, *extra):
    command = [sys.executable, "-m", module, stage, "--model", str(MODEL), "--device", "cuda:0"]
    subprocess.run([*command, "--out", str(OUT / out), *extra], cwd=CODE, check=True)
    print(stage, out, (OUT / out).read_text().splitlines()[-1], flush=True)


subprocess.run(
    [sys.executable, "-m", "octlm.day6", "prepare", "--model", str(MODEL)], cwd=CODE, check=True
)
run("octlm.day8", "train", "sft-1.jsonl", "--seed", "1", "--adapter", str(SFT))
run("octlm.day8", "eval", "eval-sft.jsonl", "--adapter", str(SFT))
train_tasks = str(CODE / "fixtures/day8/tasks.json")
run(
    "octlm.day9",
    "probe",
    "probe-sft-tasks.jsonl",
    "--adapter",
    str(SFT),
    "--tasks",
    train_tasks,
    "--limit",
    "12",
)
run("octlm.day9", "probe", "probe-before.jsonl", "--adapter", str(SFT))
run("octlm.day9", "train", "grpo.jsonl", "--adapter", str(SFT), "--save", str(GRPO), "--hours", "5")
run("octlm.day9", "probe", "probe-after.jsonl", "--adapter", str(GRPO))
run("octlm.day8", "eval", "eval-grpo.jsonl", "--adapter", str(GRPO))